# Unified Evaluation Notebook

Evaluates generated audio from any model variant by pointing at a folder of samples.

## Expected folder structure

**4-stem models** (MSDM, MSG-LD, MSLDM):
```
gen/
├── sample_00001/
│   ├── bass.wav
│   ├── drums.wav
│   ├── guitar.wav
│   ├── piano.wav
│   └── mix.wav
└── sample_00002/
    └── ...
```

**2-component models** (MGE-LDM):
```
gen/
├── sample_00001/
│   ├── src.wav
│   ├── submix.wav
│   └── mix.wav
└── sample_00002/
    └── ...
```

The reference folder (`ref/`) must mirror the same structure.

Model type is auto-detected from the filenames in the first sample directory.

In [ ]:
!pip install -q "numpy==1.23.4"
!pip install -q 'Cython<3' wheel
!pip install madmom --no-build-isolation
!pip install -q frechet-audio-distance laion-clap soundfile librosa tqdm torchvision torchmetrics torchvggish beat-this

In [1]:
import collections.abc
collections.MutableSequence = collections.abc.MutableSequence
collections.Callable = collections.abc.Callable
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping
collections.Iterable = collections.abc.Iterable
collections.Iterator = collections.abc.Iterator

import numpy as np
np.float = float
np.int = int
np.complex = complex
np.object = object
np.bool = bool

In [3]:
import sys
import warnings
import json
import datetime
import argparse as _argparse
from pathlib import Path
from typing import Optional

import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm
import torch
import shutil


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

if not getattr(torch.load, "_weights_only_patched", False):
    _real_torch_load = torch.load
    def _patched_torch_load(*a, **kw):
        return _real_torch_load(*a, **{**kw, "weights_only": False})
    _patched_torch_load._weights_only_patched = True
    torch.load = _patched_torch_load

_orig_parse_args = _argparse.ArgumentParser.parse_args

def _permissive_parse_args(self, args=None, namespace=None):
    try:
        return _orig_parse_args(self, args, namespace)
    except SystemExit:
        ns, _ = self.parse_known_args(args, namespace)
        return ns

_argparse.ArgumentParser.parse_args = _permissive_parse_args

_laion_clap_available = False
try:
    import laion_clap as _laion_clap
    if not getattr(_laion_clap.CLAP_Module.get_audio_embedding_from_data,
                   "_no_grad_patched", False):
        _orig_clap_embed = _laion_clap.CLAP_Module.get_audio_embedding_from_data
        def _patched_clap_embed(self, x, use_tensor=False, **kw):
            if torch.is_tensor(x):
                x = x.detach().cpu().numpy()
            with torch.no_grad():
                result = _orig_clap_embed(self, x, **kw)
            if use_tensor:
                result = torch.from_numpy(result)
            return result
        _patched_clap_embed._no_grad_patched = True
        _laion_clap.CLAP_Module.get_audio_embedding_from_data = _patched_clap_embed
    _laion_clap_available = True
except ImportError:
    pass

_argparse.ArgumentParser.parse_args = _orig_parse_args

_fad_lib_available = False
try:
    _orig_argv = sys.argv
    sys.argv = sys.argv[:1]
    from frechet_audio_distance import FrechetAudioDistance
    sys.argv = _orig_argv
    _fad_lib_available = True
except ImportError:
    pass

sys.path.insert(0, str(Path(".").resolve()))
from mgs_evals import (
    SISDR, MelMSE,
    FAD, KAD,
    IRS, CBS, CBD,
    BeatAlignment, COCOLA,
    VGGishEmbedder, CLAPEmbedder,
)

print(f"Imports OK  (laion_clap: {_laion_clap_available}, fad_lib: {_fad_lib_available})")

Imports OK  (laion_clap: True, fad_lib: True)


In [4]:
ROOT = Path("inference/MSG-LD/")

GEN_DIR = Path(ROOT / "gen/")
SEP_DIR = Path(ROOT / "sep/")
REF_DIR = Path("data/slakh2100/ref_mix_15")

TMP_DIR = Path("tmp/unified_eval/")
EMB_CACHE = TMP_DIR / "embeddings"  # cached .npy embeddings for FAD-lib reuse

COCOLA_CKPT = "path/to/cocola.ckpt"

COMPUTE_FAD     = True   # FAD-VGGish and FAD-CLAP (via mgs_evals embedders)
COMPUTE_KAD     = True   # KAD-VGGish (via mgs_evals embedders)
COMPUTE_FAD_LIB = True   # FAD-VGGish and FAD-CLAP (via frechet-audio-distance library)
COMPUTE_IRS     = True   # Intra-Rhythmic Stability
COMPUTE_CBS     = True   # Cross-track Beat Synchronization
COMPUTE_CBD     = True   # Cross-track Beat Dispersion
COMPUTE_BA      = True   # Beat Alignment (F-measure)
COMPUTE_COCOLA  = True   # Stem-accompaniment coherence
COMPUTE_SISDR   = True   # SI-SDR  (auto-skipped for MGE-LDM)
COMPUTE_MEL_MSE = True   # Mel-MSE (auto-skipped for MGE-LDM)

DEVICE     = "cpu"   # "cuda" if a GPU is available
SR         = 16000
SR_CLAP    = 48000   # CLAP expects 48 kHz
N_MELS     = 64
BATCH_SIZE = 32      # for FAD/KAD embedding

results = {}

## Create Reference Segments (optional, run once)

Segments the Slakh2100 test tracks into fixed-length clips that become `REF_DIR`

In [ ]:
import torchaudio as _torchaudio

SLAKH_TEST_DIR = Path("data/slakh2100/full/test")       # source Slakh2100 test tracks
SEG_SEC        = 15.0                                   # clip length in seconds
SLAKH_STEMS    = ["bass", "drums", "guitar", "piano"]
SR_SEG         = 16000                                  # output sample rate
REF_MAX        = 500                                    # keep this many segments after generation; None = keep all

print(f"Source  : {SLAKH_TEST_DIR}")
print(f"Output  : {REF_DIR}  ({SEG_SEC}s @ {SR_SEG} Hz)")
print(f"Stems   : {SLAKH_STEMS}")
print(f"REF_MAX : {REF_MAX}")

_n_existing = sum(1 for d in REF_DIR.iterdir() if d.is_dir()) if REF_DIR.exists() else 0

if _n_existing > 0:
    print()
    print(f"Found {_n_existing} existing segments in {REF_DIR}. skipping.")
    print(f"Delete the folder to re-run segmentation from scratch.")
else:
    if not SLAKH_TEST_DIR.exists():
        raise FileNotFoundError(f"Slakh test directory not found: {SLAKH_TEST_DIR}")

    _clip_len   = int(SEG_SEC * SR_SEG)
    _track_dirs = sorted(d for d in SLAKH_TEST_DIR.iterdir() if d.is_dir())
    REF_DIR.mkdir(parents=True, exist_ok=True)
    _n_written  = 0

    for _track in tqdm(_track_dirs, desc="Segmenting"):
        _stem_audio = {}
        for _stem in SLAKH_STEMS:
            _fpath = _track / f"{_stem}.wav"
            if not _fpath.exists():
                continue
            _audio, _file_sr = sf.read(str(_fpath), dtype="float32", always_2d=False)
            if _audio.ndim == 2:
                _audio = _audio.mean(axis=1)
            if _file_sr != SR_SEG:
                _wav   = torch.from_numpy(_audio).unsqueeze(0)
                _audio = _torchaudio.functional.resample(_wav, _file_sr, SR_SEG).squeeze(0).numpy()
            _stem_audio[_stem] = _audio

        if not _stem_audio:
            continue

        _n_samples = min(len(a) for a in _stem_audio.values())
        _n_segs    = _n_samples // _clip_len

        for _seg_idx in range(_n_segs):
            _start   = _seg_idx * _clip_len
            _seg_dir = REF_DIR / f"{_track.name}_seg{_seg_idx:04d}"
            _seg_dir.mkdir(exist_ok=True)

            for _stem, _a in _stem_audio.items():
                sf.write(str(_seg_dir / f"{_stem}.wav"), _a[_start:_start + _clip_len], SR_SEG)

            _mix = sum(_stem_audio[s][_start:_start + _clip_len] for s in _stem_audio)
            sf.write(str(_seg_dir / "mix.wav"), _mix, SR_SEG)
            _n_written += 1

    print(f"Wrote {_n_written} segments to {REF_DIR}")

    if REF_MAX is not None and _n_written > REF_MAX:
        _all_seg_dirs = sorted(d for d in REF_DIR.iterdir() if d.is_dir())
        _rng = np.random.default_rng(42)
        _keep_idx = set(_rng.choice(len(_all_seg_dirs), REF_MAX, replace=False).tolist())
        _n_deleted = 0
        for _i, _d in enumerate(_all_seg_dirs):
            if _i not in _keep_idx:
                shutil.rmtree(_d)
                _n_deleted += 1
        print(f"Kept {REF_MAX} / {_n_written} segments (deleted {_n_deleted}).")

_n_final = sum(1 for d in REF_DIR.iterdir() if d.is_dir())
print()
print(f"REF_DIR = {REF_DIR}  ({_n_final} segments)")


In [ ]:
def _detect_model_type(gen_dir):
    sample_dirs = sorted(d for d in gen_dir.iterdir() if d.is_dir())
    if not sample_dirs:
        raise ValueError(f"No sample subdirectories found in {gen_dir}")
    wav_names = {f.stem for f in sample_dirs[0].iterdir() if f.suffix == ".wav"}
    if "src" in wav_names and "submix" in wav_names:
        return "mge_ldm", ["src", "submix"]
    return "4stem", ["bass", "drums", "guitar", "piano"]


MODEL_TYPE, STEMS = _detect_model_type(GEN_DIR)

_first_sample = sorted(d for d in GEN_DIR.iterdir() if d.is_dir())[0]
HAS_MIX = (_first_sample / "mix.wav").exists()
N_SAMPLES = sum(1 for d in GEN_DIR.iterdir() if d.is_dir())

print(f"Model type : {MODEL_TYPE}")
print(f"Stems      : {STEMS}")
print(f"Samples    : {N_SAMPLES}")
print(f"Has mix.wav: {HAS_MIX}")

if MODEL_TYPE == "mge_ldm":
    print()
    print("MGE-LDM detected: SI-SDR and Mel-MSE will be skipped.")
    COMPUTE_SISDR   = False
    COMPUTE_MEL_MSE = False

In [ ]:
def collect_stem_files(root, stem):
    files = []
    for sample_dir in sorted(d for d in root.iterdir() if d.is_dir()):
        fpath = sample_dir / f"{stem}.wav"
        if fpath.is_file():
            files.append(str(fpath))
    return files


def stage_flat_dir(files, dst_dir):
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(files):
        dst = dst_dir / f"{i:06d}.wav"
        if not dst.exists():
            try:
                dst.symlink_to(Path(src).resolve())
            except (OSError, NotImplementedError):
                shutil.copy2(str(src), str(dst))


def rms_normalize(audio, target=0.1):
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-8:
        return audio
    return audio * (target / rms)


def load_audio_mono(path, sr=SR):
    audio, file_sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if file_sr != sr:
        audio = librosa.resample(audio, orig_sr=file_sr, target_sr=sr)
    return audio


def audio_to_mel(audio, sr=SR, n_mels=N_MELS):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    return librosa.power_to_db(mel, ref=np.max)


print("Utilities defined.")

## FAD + KAD

For 4-stem models: evaluated per stem and on the mixture.

For MGE-LDM: mixture only.

In [ ]:
if COMPUTE_FAD or COMPUTE_KAD:
    if MODEL_TYPE == "mge_ldm":
        fad_kad_channels = ["mix"] if HAS_MIX else []
    else:
        fad_kad_channels = STEMS[:] + (["mix"] if HAS_MIX else [])

    print(f"FAD/KAD channels: {fad_kad_channels}")

    _vggish = VGGishEmbedder(device=DEVICE)
    _fad_vgg = FAD(backend="vggish", embedder=_vggish)
    _kad_vgg = KAD(backend="vggish", embedder=_vggish)

    if COMPUTE_FAD:
        _clap = CLAPEmbedder(device=DEVICE)
        _fad_clap = FAD(backend="clap", embedder=_clap)

    for _ch in fad_kad_channels:
        _gen_files = collect_stem_files(GEN_DIR, _ch)
        _ref_files = collect_stem_files(REF_DIR, _ch)

        if not _gen_files:
            print(f"[{_ch}] No generated files found, skipping.")
            continue
        if not _ref_files:
            print(f"[{_ch}] No reference files found, skipping.")
            continue

        print()
        print(f"Gen : {len(_gen_files)} files")
        print(f"Ref : {len(_ref_files)} files")
        print(f"[{_ch}] Embedding VGGish")
        _gen_vgg = _vggish.embed_files(_gen_files, sr=SR, batch_size=BATCH_SIZE)
        _ref_vgg = _vggish.embed_files(_ref_files, sr=SR, batch_size=BATCH_SIZE)

        if COMPUTE_FAD:
            _r = _fad_vgg.compute(gen_embeddings=_gen_vgg, ref_embeddings=_ref_vgg)
            results[f"fad_vggish_{_ch}"] = _r["fad_vggish"]
            print(f"  FAD-VGGish : {_r['fad_vggish']:.4f}")

        if COMPUTE_KAD:
            _r = _kad_vgg.compute(gen_embeddings=_gen_vgg, ref_embeddings=_ref_vgg)
            results[f"kad_vggish_{_ch}"] = _r["kad_vggish"]
            print(f"  KAD-VGGish : {_r['kad_vggish']:.4f}")

        if COMPUTE_FAD:
            print(f"[{_ch}] Embedding CLAP")
            _gen_clap = _clap.embed_files(_gen_files, sr=SR_CLAP, batch_size=BATCH_SIZE // 2)
            _ref_clap = _clap.embed_files(_ref_files, sr=SR_CLAP, batch_size=BATCH_SIZE // 2)
            _r = _fad_clap.compute(gen_embeddings=_gen_clap, ref_embeddings=_ref_clap)
            results[f"fad_clap_{_ch}"] = _r["fad_clap"]
            print(f"  FAD-CLAP   : {_r['fad_clap']:.4f}")

    print()
    print("FAD/KAD done.")
else:
    print("FAD/KAD skipped (both COMPUTE_FAD and COMPUTE_KAD are False).")

## FAD - `frechet-audio-distance` library

Same VGGish and CLAP metrics as above, computed via the `frechet-audio-distance`
library (`FrechetAudioDistance.score()`).

In [ ]:
if COMPUTE_FAD_LIB:
    if not _fad_lib_available:
        print("frechet-audio-distance not installed - skipping FAD-lib.")
    else:
        if MODEL_TYPE == "mge_ldm":
            _fad_lib_channels = ["mix"] if HAS_MIX else []
        else:
            _fad_lib_channels = STEMS[:] + (["mix"] if HAS_MIX else [])

        print(f"FAD-lib channels: {_fad_lib_channels}")
        EMB_CACHE.mkdir(parents=True, exist_ok=True)
        _fad_lib_tmp = TMP_DIR / "fad_lib"

        _frechet_vgg = FrechetAudioDistance(
            model_name="vggish",
            sample_rate=16000,
            use_pca=False,
            use_activation=False,
            verbose=False,
        )
        _frechet_clap = FrechetAudioDistance(
            model_name="clap",
            sample_rate=48000,
            submodel_name="630k-audioset",
            verbose=False,
            enable_fusion=False,
        )

        for _ch in _fad_lib_channels:
            _gen_files = collect_stem_files(GEN_DIR, _ch)
            _ref_files = collect_stem_files(REF_DIR, _ch)

            if not _gen_files or not _ref_files:
                print(f"[{_ch}] No files found, skipping.")
                continue

            _gen_flat = _fad_lib_tmp / "gen" / _ch
            _ref_flat = _fad_lib_tmp / "ref" / _ch
            print()
            print(f"Gen : {len(_gen_files)} files")
            print(f"Ref : {len(_ref_files)} files")
            print(f"[{_ch}] Staging")
            stage_flat_dir(_gen_files, _gen_flat)
            stage_flat_dir(_ref_files, _ref_flat)

            _score_vgg = _frechet_vgg.score(
                str(_ref_flat), str(_gen_flat), dtype="float32",
                background_embds_path=str(EMB_CACHE / f"vgg_ref_{_ch}.npy"),
                eval_embds_path=str(EMB_CACHE / f"vgg_gen_{_ch}.npy"),
            )
            results[f"fad_lib_vggish_{_ch}"] = _score_vgg
            print(f"  FAD-lib VGGish : {_score_vgg:.4f}")

            _score_clap = _frechet_clap.score(
                str(_ref_flat), str(_gen_flat), dtype="float32",
                background_embds_path=str(EMB_CACHE / f"clap_ref_{_ch}.npy"),
                eval_embds_path=str(EMB_CACHE / f"clap_gen_{_ch}.npy"),
            )
            results[f"fad_lib_clap_{_ch}"] = _score_clap
            print(f"  FAD-lib CLAP   : {_score_clap:.4f}")

        print()
        print("FAD-lib done.")
else:
    print("FAD-lib skipped.")

## IRS

In [ ]:
if COMPUTE_IRS:
    _irs_result = IRS().compute(folder=str(GEN_DIR), stems=STEMS, max_amount=50)
    results["irs"] = _irs_result["irs"]
    results["irs_stem_results"] = _irs_result["irs_stem_results"]

    print(f"IRS overall : {_irs_result['irs']:.4f}   (lower = more stable)")
    print("Per-stem CV:")
    for _stem, _stats in _irs_result["irs_stem_results"].items():
        print(f"  {_stem:10s}: {_stats['avg_cv']:.4f}  (n={_stats['file_count']})")
else:
    print("IRS skipped.")

## CBS + CBD

In [ ]:
if COMPUTE_CBS:
    print("Computing CBS")
    _cbs_result = CBS().compute(folder=str(GEN_DIR), stems=STEMS)
    results["cbs"] = _cbs_result["cbs_mean_beat_ratio"]
    print(f"CBS : {_cbs_result['cbs_mean_beat_ratio']:.4f}   (higher = better)")
else:
    print("CBS skipped.")

if COMPUTE_CBD:
    print()
    print("Computing CBD")
    _cbd_result = CBD().compute(folder=str(GEN_DIR), stems=STEMS)
    results["cbd_mean"]   = _cbd_result["cbd_avg_error"]
    results["cbd_std"]    = _cbd_result["cbd_avg_std_error"]
    results["cbd_median"] = _cbd_result["cbd_avg_median_error"]
    print(f"CBD mean   : {_cbd_result['cbd_avg_error']:.4f}   (lower = better)")
    print(f"CBD std    : {_cbd_result['cbd_avg_std_error']:.4f}")
    print(f"CBD median : {_cbd_result['cbd_avg_median_error']:.4f}")
else:
    print("CBD skipped.")

## Beat Alignment

In [ ]:
if COMPUTE_BA:
    print("Computing Beat Alignment")
    _ba_result = BeatAlignment().compute(folder=str(GEN_DIR), stems=STEMS)
    results["ba_fmeasure"]      = _ba_result["ba_fmeasure"]
    results["ba_stem_fmeasure"] = _ba_result["ba_stem_fmeasure"]

    print(f"BA F-measure : {_ba_result['ba_fmeasure']:.4f}   (higher = better)")
    print(f"Skipped      : {_ba_result['ba_n_skipped']}")
    print("Per-stem F-measure:")
    for _stem, _fm in _ba_result["ba_stem_fmeasure"].items():
        print(f"  {_stem:10s}: {_fm:.4f}")
else:
    print("Beat Alignment skipped.")

## COCOLA

Set `COCOLA_CKPT` in the configuration cell before running this.

In [ ]:
if COMPUTE_COCOLA:
    if COCOLA_CKPT == "path/to/cocola.ckpt" or not Path(COCOLA_CKPT).is_file():
        print("COCOLA skipped: set COCOLA_CKPT to the checkpoint path in the config cell.")
    else:
        print("Computing COCOLA")
        _cocola_result = COCOLA(checkpoint_path=COCOLA_CKPT, device=DEVICE).compute(
            folder=str(GEN_DIR), stems=STEMS
        )
        results["cocola_both"]       = _cocola_result["cocola_both"]
        results["cocola_harmonic"]   = _cocola_result["cocola_harmonic"]
        results["cocola_percussive"] = _cocola_result["cocola_percussive"]

        print(f"COCOLA both       : {_cocola_result['cocola_both']:.4f}  "
              f"(random: {_cocola_result['cocola_random_both']:.4f})")
        print(f"COCOLA harmonic   : {_cocola_result['cocola_harmonic']:.4f}  "
              f"(random: {_cocola_result['cocola_random_harmonic']:.4f})")
        print(f"COCOLA percussive : {_cocola_result['cocola_percussive']:.4f}  "
              f"(random: {_cocola_result['cocola_random_percussive']:.4f})")
else:
    print("COCOLA skipped.")

## SI-SDR + Mel-MSE

Paired evaluation

Automatically skipped for MGE-LDM.

In [ ]:
if COMPUTE_SISDR or COMPUTE_MEL_MSE:
    if not SEP_DIR.exists():
        print(f"SEP_DIR not found: {SEP_DIR}")
    else:
        _track_dirs = sorted(d for d in SEP_DIR.iterdir() if d.is_dir())
        print(f"{len(_track_dirs)} tracks in {SEP_DIR}")

        _sisdr_metric   = SISDR() if COMPUTE_SISDR else None
        _mel_mse_metric = MelMSE() if COMPUTE_MEL_MSE else None

        _sep_sisdr   = {s: [] for s in STEMS}
        _sep_mel_mse = {s: [] for s in STEMS}

        for _td in tqdm(_track_dirs, desc="Separation eval"):
            for _stem in STEMS:
                _est_path = _td / "sep"  / f"{_stem}.wav"
                _ref_path = _td / "orig" / f"{_stem}.wav"
                if not _est_path.is_file() or not _ref_path.is_file():
                    continue

                _est = load_audio_mono(str(_est_path))
                _ref = load_audio_mono(str(_ref_path))
                _n = min(len(_est), len(_ref))
                _est, _ref = _est[:_n], _ref[:_n]

                if COMPUTE_SISDR:
                    _r = _sisdr_metric.compute(estimates=_est, references=_ref)
                    _sep_sisdr[_stem].append(_r["si_sdr"])

                if COMPUTE_MEL_MSE:
                    _est_mel = audio_to_mel(rms_normalize(_est))
                    _ref_mel = audio_to_mel(rms_normalize(_ref))
                    _t = min(_est_mel.shape[-1], _ref_mel.shape[-1])
                    _r = _mel_mse_metric.compute(
                        estimates_mel=_est_mel[:, :_t], references_mel=_ref_mel[:, :_t]
                    )
                    _sep_mel_mse[_stem].append(_r["mel_mse"])

        if COMPUTE_SISDR:
            print()
            print("SI-SDR (dB, higher is better):")
            _all_sisdr = []
            for _stem in STEMS:
                _vals = [v for v in _sep_sisdr[_stem] if not np.isnan(v)]
                if _vals:
                    _mean = float(np.mean(_vals))
                    results[f"sisdr_{_stem}"] = _mean
                    _all_sisdr.extend(_vals)
                    print(f"  {_stem:10s}: {_mean:.2f} dB")
            if _all_sisdr:
                results["sisdr_mean"] = float(np.mean(_all_sisdr))
                print(f"  {'mean':10s}: {results['sisdr_mean']:.2f} dB")

        if COMPUTE_MEL_MSE:
            print()
            print("Mel-MSE (lower is better):")
            _all_melmse = []
            for _stem in STEMS:
                _vals = [v for v in _sep_mel_mse[_stem] if not np.isnan(v)]
                if _vals:
                    _mean = float(np.mean(_vals))
                    results[f"mel_mse_{_stem}"] = _mean
                    _all_melmse.extend(_vals)
                    print(f"  {_stem:10s}: {_mean:.6f}")
            if _all_melmse:
                results["mel_mse_mean"] = float(np.mean(_all_melmse))
                print(f"  {'mean':10s}: {results['mel_mse_mean']:.6f}")
else:
    print("SI-SDR and Mel-MSE skipped.")

## Results Summary

In [ ]:
print(f"Evaluation results  ({MODEL_TYPE}, {N_SAMPLES} samples)")

def _fmt(v):
    if isinstance(v, float):
        return f"{v:.4f}"
    return str(v)

_skip = {"irs_stem_results", "ba_stem_fmeasure"}

for _k, _v in results.items():
    if _k in _skip:
        continue
    print(f"  {_k:<30s} {_fmt(_v)}")

print()
if "irs_stem_results" in results:
    print("  IRS per stem:")
    for _s, _st in results["irs_stem_results"].items():
        print(f"    {_s:10s}: {_st['avg_cv']:.4f}")

if "ba_stem_fmeasure" in results:
    print("  BA per stem:")
    for _s, _fm in results["ba_stem_fmeasure"].items():
        print(f"    {_s:10s}: {_fm:.4f}")

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_out_path = Path("results") / f"eval_{MODEL_TYPE}_{_ts}.json"
_out_path.parent.mkdir(exist_ok=True)

_json_safe = {
    k: v for k, v in results.items()
    if not isinstance(v, list)
}
with open(_out_path, "w") as _f:
    json.dump({"model_type": MODEL_TYPE, "n_samples": N_SAMPLES, **_json_safe}, _f, indent=2)

print()
print(f"Saved to {_out_path}")